# Learning to classically simulate RCS from samples

## Parameters

In [8]:
# Experiment parameters.
N_VALUES = [12, 16, 20, 24]  # Number of qubits.

# Circuit parameters.
CZ_DEPTH = 10
SEED     = 42

# Model parameters/options.
USE_PT_REGULARISATION = False   # Set True to penalise non-Porter-Thomas distributions.
LAMBDA_PT             = 0.1     # Regularisation strength (only used if above is True).

# Training parameters.
BATCH_SIZE   = 512
TOTAL_STEPS  = 50_000   # fixed gradient-step budget for all N_train sizes
MIN_EPOCHS   = 50       # always do at least this many epochs regardless of dataset size
MAX_EPOCHS   = 5_000    # cap for tiny datasets to keep runtime sane
N_TEST_OVERLAP = 10_000  # samples used for overlap / generalisation diagnostics


## Setup

### Imports

In [9]:
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import cirq
import matplotlib.pyplot as plt
from typing import Iterable, Callable, Sequence, TypeVar, cast

### Seed and device

In [10]:
torch.manual_seed(0)
np.random.seed(0)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DEVICE

'cpu'

### Circuit definition

In [11]:
# Source: https://github.com/quantumlib/ReCirq/blob/main/recirq/beyond_classical/google_v2_beyond_classical.py
T = TypeVar('T')

def _choice(rand_gen: Callable[[], float], sequence: Sequence[T]) -> T:
    return sequence[int(rand_gen() * len(sequence))]

def _make_cz_layer(qubits: Iterable[cirq.GridQubit], layer_index: int):
    layer_index_map = [0, 3, 2, 1, 4, 7, 6, 5]
    internal_layer_index = layer_index_map[layer_index % 8]
    dir_row = internal_layer_index % 2
    dir_col = 1 - dir_row
    shift   = (internal_layer_index >> 1) % 4
    for q in qubits:
        q2 = cirq.GridQubit(q.row + dir_row, q.col + dir_col)
        if q2 not in qubits:
            continue
        if (q.row * (2 - dir_row) + q.col * (2 - dir_col)) % 4 != shift:
            continue
        yield cirq.CZ(q, q2)

def _add_cz_layer(layer_index: int, circuit: cirq.Circuit) -> int:
    cz_layer = None
    while not cz_layer:
        qubits   = cast(Iterable[cirq.GridQubit], circuit.all_qubits())
        cz_layer = list(_make_cz_layer(qubits, layer_index))
        layer_index += 1
    circuit.append(cz_layer, strategy=cirq.InsertStrategy.NEW_THEN_INLINE)
    return layer_index

def generate_rcs_circuit(qubits: Iterable[cirq.GridQubit], cz_depth: int, seed: int) -> cirq.Circuit:
    """Google v2 RCS circuit (Boixo et al. 2018)."""
    non_diagonal_gates = [cirq.X ** (1/2), cirq.Y ** (1/2)]
    rand_gen = random.Random(seed).random
    circuit  = cirq.Circuit()

    circuit.append(cirq.H(q) for q in qubits)

    layer_index = 0
    if cz_depth:
        layer_index = _add_cz_layer(layer_index, circuit)
        for q in qubits:
            if not circuit.operation_at(q, 1):
                circuit.append(cirq.T(q), strategy=cirq.InsertStrategy.EARLIEST)
        for moment_index in range(2, cz_depth + 1):
            layer_index = _add_cz_layer(layer_index, circuit)
            for q in qubits:
                if not circuit.operation_at(q, moment_index):
                    last_op = circuit.operation_at(q, moment_index - 1)
                    if last_op:
                        gate = cast(cirq.GateOperation, last_op).gate
                        if gate == cirq.CZ:
                            circuit.append(
                                _choice(rand_gen, non_diagonal_gates).on(q),
                                strategy=cirq.InsertStrategy.EARLIEST)
                        elif gate != cirq.T:
                            circuit.append(cirq.T(q), strategy=cirq.InsertStrategy.EARLIEST)

    circuit.append([cirq.H(q) for q in qubits], strategy=cirq.InsertStrategy.NEW_THEN_INLINE)
    return circuit

def grid_dimensions(n_qubits):
    """Return (n_rows, n_cols) for a roughly-square grid with n_rows <= n_cols."""
    for n_rows in range(2, n_qubits + 1):
        if n_qubits % n_rows == 0:
            return n_rows, n_qubits // n_rows
    return 1, n_qubits

def generate_rcs_grid(n_qubits: int, cz_depth: int, seed: int) -> cirq.Circuit:
    n_rows, n_cols = grid_dimensions(n_qubits)
    qubits = [cirq.GridQubit(i, j) for i in range(n_rows) for j in range(n_cols)]
    return generate_rcs_circuit(qubits, cz_depth, seed)

### Circuit simulation functions

In [12]:
def simulate_circuit(circuit: cirq.Circuit) -> np.ndarray:
    """Return exact probability distribution over all 2^n bitstrings."""
    sv    = cirq.Simulator().simulate(circuit).final_state_vector
    probs = np.abs(sv) ** 2
    return probs / probs.sum()

def index_to_bitstring(idx: int, n_bits: int) -> np.ndarray:
    """Integer index → binary array, MSB first."""
    return np.array([(idx >> (n_bits - 1 - i)) & 1 for i in range(n_bits)], dtype=np.float32)

def bitstrings_to_indices(bits: np.ndarray) -> np.ndarray:
    """(N, n_bits) binary array → integer indices."""
    powers = 2 ** np.arange(bits.shape[1] - 1, -1, -1)
    return (bits @ powers).astype(int)

def xeb_fidelity(samples: np.ndarray, probs: np.ndarray) -> float:
    """Linear XEB fidelity: N*<p(x)>_samples - 1.  Ideal~1, uniform~0."""
    return float(len(probs) * probs[bitstrings_to_indices(samples)].mean() - 1)

def kl_divergence(model_probs: np.ndarray, true_probs: np.ndarray, eps: float = 1e-12) -> float:
    """KL(true || model). Only feasible for small n."""
    return float(np.sum(true_probs * np.log((true_probs + eps) / (model_probs + eps))))

def sample_bitstrings(probs: np.ndarray, n_bits: int, n_samples: int, seed: int = None) -> np.ndarray:
    """Draw n_samples bitstrings from distribution probs."""
    rng = np.random.default_rng(seed)
    idx = rng.choice(len(probs), size=n_samples, p=probs)
    return np.stack([index_to_bitstring(i, n_bits) for i in idx])

### Autoregressive RNN Model

Models $p(x) = p(x_1) · p(x_2|x_1) \cdots p(x_n|x_1,…,x_{n - 1})$ via a two-layer LSTM.

In [13]:
class AutoregressiveRNN(nn.Module):
    """LSTM-based autoregressive model over binary strings."""

    def __init__(self, n_bits: int, hidden: int = 128, n_layers: int = 2):
        super().__init__()
        self.n_bits   = n_bits
        self.lstm     = nn.LSTM(1, hidden, n_layers, batch_first=True)
        self.head     = nn.Linear(hidden, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x: (B, n_bits) → logits: (B, n_bits)."""
        bsz = x.shape[0]
        sos = torch.zeros(bsz, 1, 1, device=x.device)
        inp = torch.cat([sos, x[:, :-1].unsqueeze(-1)], dim=1)
        out, _ = self.lstm(inp)
        return self.head(out).squeeze(-1)

    def log_prob(self, x: torch.Tensor) -> torch.Tensor:
        """Sum of per-bit log-probs → log p(x)."""
        logits = self.forward(x)
        return -nn.functional.binary_cross_entropy_with_logits(
            logits, x, reduction='none').sum(dim=1)

    @torch.no_grad()
    def sample_bits(self, n: int) -> np.ndarray:
        """Autoregressive sampling — one bit at a time."""
        device  = next(self.parameters()).device
        samples = torch.zeros(n, self.n_bits, device=device)
        h       = None
        inp     = torch.zeros(n, 1, 1, device=device)
        for i in range(self.n_bits):
            out, h = self.lstm(inp, h)
            prob   = torch.sigmoid(self.head(out.squeeze(1)))
            bit    = torch.bernoulli(prob)
            samples[:, i] = bit.squeeze(1)
            inp = bit.unsqueeze(1)
        return samples.cpu().numpy()

    @torch.no_grad()
    def full_distribution(self, n_states: int, n_bits: int) -> np.ndarray:
        """p(x) for all 2^n_bits bitstrings. Only feasible for small n."""
        device   = next(self.parameters()).device
        all_bits = np.array([index_to_bitstring(i, n_bits) for i in range(n_states)],
                             dtype=np.float32)
        x  = torch.tensor(all_bits, device=device)
        lp = self.log_prob(x).cpu().numpy()
        p  = np.exp(lp - lp.max())
        return p / p.sum()


def train_rnn(n_bits, probs, n_train, total_steps=TOTAL_STEPS, min_epochs=MIN_EPOCHS, max_epochs=MAX_EPOCHS, hidden=128, seed=1):
    """
    Train a fresh RNN with a fixed gradient-step budget.

    Metrics returned:
      xeb_model    : XEB of model-generated samples (existing metric)
      xeb_gen      : Generalisation XEB — model probabilities scored on
                     held-out quantum samples the model never saw.
                     F_gen = N * <q(z)>_{z ~ p_U} - 1
                     where q is the model distribution, z are fresh quantum samples.
                     This is the correct test of whether the model has learned p_U,
                     not just memorised the training set.
      kl           : KL(true || model), only for n_bits <= 16
    """
    torch.manual_seed(seed); np.random.seed(seed)

    # Training samples (seed=seed) and held-out test samples (seed=seed+999)
    n_states   = len(probs)
    train_bits = sample_bitstrings(probs, n_bits, n_train,   seed=seed)
    test_bits  = sample_bitstrings(probs, n_bits, 10_000,    seed=seed + 999)

    loader = DataLoader(TensorDataset(torch.tensor(train_bits)),
                        batch_size=min(BATCH_SIZE, n_train), shuffle=True)

    steps_per_epoch = max(1, n_train // BATCH_SIZE)
    n_epochs        = min(max_epochs, max(min_epochs, total_steps // steps_per_epoch))

    model     = AutoregressiveRNN(n_bits=n_bits, hidden=hidden).to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs)
    losses    = []

    for epoch in range(1, n_epochs + 1):
        model.train()
        epoch_loss = 0.0
        for (batch,) in loader:
            batch = batch.to(DEVICE)
            nll_loss = -model.log_prob(batch).mean()

            if USE_PT_REGULARISATION:
                # Porter-Thomas regulariser: push marginal distribution of N*q(z)
                # toward Exponential(1).  KL(Exp(1) || q_marginal) ∝ E[q - log q]
                # evaluated on the current batch's model probabilities.
                with torch.no_grad():
                    log_q_batch = model.log_prob(batch)
                log_q_batch  = model.log_prob(batch)   # need grad
                q_scaled     = torch.exp(log_q_batch) * n_states  # N*q(z), mean~1 under PT
                pt_loss      = (q_scaled - log_q_batch).mean()      # E[q - log q]
                loss         = nll_loss + LAMBDA_PT * pt_loss
            else:
                loss = nll_loss

            optimizer.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            epoch_loss += loss.item()
        scheduler.step()
        losses.append(epoch_loss / len(loader))
        print("\r", end=f"Current loss: {losses[-1]}")

    model.eval()

    # XEB of model-generated samples (original metric)
    model_samples = model.sample_bits(10_000)
    xeb_model     = xeb_fidelity(model_samples, probs)

    # Generalisation XEB: score model probabilities on held-out quantum samples
    # F_gen = N * <q(z)>_{z ~ p_U} - 1  where q(z) = model probability of z
    test_t        = torch.tensor(test_bits, device=DEVICE)
    with torch.no_grad():
        log_q     = model.log_prob(test_t).cpu().numpy()
    q_probs       = np.exp(log_q)                        # model prob of each test bitstring
    xeb_gen       = float(len(probs) * q_probs.mean() - 1)

    kl = None
    if n_bits <= 16:
        kl = kl_divergence(model.full_distribution(len(probs), n_bits), probs)

    return {'n_bits': n_bits, 'n_train': n_train, 'n_epochs': n_epochs,
            'xeb_model': xeb_model, 'xeb_gen': xeb_gen,
            'kl': kl, 'final_nll': losses[-1], 'losses': losses,
            '_model_samples': model_samples,   # kept for diagnostics, not logged
            '_train_bits':    train_bits,
            '_test_bits':     test_bits}

def generalisation_report(res: dict, probs: np.ndarray) -> dict:
    """
    Quantify memorisation vs generalisation using three sample sets:
      - train set  : bitstrings the model trained on
      - test set   : fresh quantum samples, never seen by model  (z ~ p_U)
      - model set  : fresh samples generated by the trained model (z ~ q)

    Key comparisons:
      overlap(model, train) high + overlap(model, test) low  → memorisation
      overlap(model, test)  comparable to overlap(test, test) → generalisation
    """
    from collections import Counter

    train_bits    = res['_train_bits']
    test_bits     = res['_test_bits']
    model_samples = res['_model_samples']

    train_idx = set(bitstrings_to_indices(train_bits).tolist())
    test_idx  = bitstrings_to_indices(test_bits).tolist()
    model_idx = bitstrings_to_indices(model_samples).tolist()

    n_model = len(model_idx)

    # Overlap: fraction of model samples that appear in train / test sets
    overlap_train = sum(1 for i in model_idx if i in train_idx) / n_model
    test_idx_set  = set(test_idx)
    overlap_test  = sum(1 for i in model_idx if i in test_idx_set) / n_model

    # Baseline: how much does the test set overlap with itself (sample from same dist)?
    # Expected ~ 1 - (1 - 1/|support|)^|test| ≈ |test|/|support| for large support
    half      = test_idx[:len(test_idx)//2]
    other     = set(test_idx[len(test_idx)//2:])
    overlap_test_self = sum(1 for i in half if i in other) / len(half)

    # Diversity: unique bitstrings in model samples
    n_unique_model = len(set(model_idx))
    top1_pct       = 100 * Counter(model_idx).most_common(1)[0][1] / n_model

    report = {
        'overlap_train':     overlap_train,
        'overlap_test':      overlap_test,
        'overlap_test_self': overlap_test_self,
        'n_unique_model':    n_unique_model,
        'top1_pct':          top1_pct,
    }

    # Verdict
    if overlap_train > 0.5 and overlap_test < overlap_test_self * 0.5:
        verdict = '⚠ MEMORISATION'
    elif overlap_test >= overlap_test_self * 0.5 and overlap_train < 0.5:
        verdict = '✓ GENERALISATION'
    elif overlap_train > 0.3:
        verdict = '~ PARTIAL MEMORISATION'
    else:
        verdict = '~ INCONCLUSIVE'
    report['verdict'] = verdict

    return report


## Experiment

In [ ]:
n_sweep_results = []
for nqubits in N_VALUES:
    print("Status: On nqubits =", nqubits)

    # Build circuit and simulate
    circuit_n = generate_rcs_grid(nqubits, CZ_DEPTH, SEED)
    probs_n   = simulate_circuit(circuit_n)
    n_states_n = len(probs_n)

    # Baselines for this n
    xeb_ideal_n  = xeb_fidelity(sample_bitstrings(probs_n, nqubits, 10_000), probs_n)
    xeb_unif_n   = xeb_fidelity(np.random.randint(0,2,(10_000,nqubits)).astype(np.float32), probs_n)
    print(f"Grid: {grid_dimensions(nqubits)} | Ideal XEB: {xeb_ideal_n:.4f} | Uniform XEB: {xeb_unif_n:.4f}")

    sweep_sizes_n = sorted(set([
        nqubits ** 2,
        int(nqubits ** 2 * np.log2(nqubits)),
        nqubits ** 3,
        int(nqubits ** 3 * np.log2(nqubits)),
        10_000,
        100_000,
        1_000_000,
    ]))

    for n_train in sweep_sizes_n:
        res = train_rnn(nqubits, probs_n, n_train=n_train)
        res.update({'xeb_ideal': xeb_ideal_n, 'xeb_unif': xeb_unif_n})
        n_sweep_results.append(res)
        kl_str = f'KL={res["kl"]:.4f}' if res['kl'] is not None else 'KL=N/A'
        print(f"N_train={n_train:>6,} ...  XEB_model={res['xeb_model']:+.4f}  XEB_gen={res['xeb_gen']:+.4f}  {kl_str}")
        gr = generalisation_report(res, probs_n)
        print(f"  overlap(model,train)={gr['overlap_train']:.3f}  overlap(model,test)={gr['overlap_test']:.3f}  overlap(test,test)={gr['overlap_test_self']:.3f}  unique={gr['n_unique_model']:,}  top1={gr['top1_pct']:.1f}%  {gr['verdict']}")

SyntaxError: f-string: unmatched '[' (1893711414.py, line 30)

In [ ]:
import pandas as pd

df = pd.DataFrame(n_sweep_results)

# Normalised XEB: 0=uniform, 1=ideal
# Two normalised metrics
df['xeb_model_norm'] = (df['xeb_model'] - df['xeb_unif']) / (df['xeb_ideal'] - df['xeb_unif'])
df['xeb_gen_norm']   = (df['xeb_gen']   - df['xeb_unif']) / (df['xeb_ideal'] - df['xeb_unif'])
# Primary metric for heatmap: generalisation XEB
df['xeb_norm'] = df['xeb_gen_norm']

n_values     = sorted(df['n_bits'].unique())
train_labels = ['n²', 'n²log n', 'n³', 'n³log n', '10k', '100k', '1M']

def poly_label(n_train, n):
    opts = {
        n**2:                    'n²',
        int(n**2 * np.log2(n)): 'n²log n',
        n**3:                    'n³',
        int(n**3 * np.log2(n)): 'n³log n',
        10_000:                  '10k',
        100_000:                 '100k',
        1_000_000:               '1M',
    }
    return opts.get(n_train, str(n_train))

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
colors = plt.cm.plasma(np.linspace(0.1, 0.9, len(train_labels)))

# ── Plot 1: XEB vs n, one curve per N_train label ─────────────────────────
for label, col in zip(train_labels, colors):
    xs, ys = [], []
    for n in n_values:
        sub = df[df['n_bits'] == n]
        for _, row in sub.iterrows():
            if poly_label(int(row['n_train']), n) == label:
                xs.append(n); ys.append(row['xeb_norm']); break
    if xs:
        axes[0].plot(xs, ys, 'o-', color=col, lw=2, ms=7, label=label)

axes[0].axhline(1.0, color='green', ls='--', lw=1.2, label='Ideal (1.0)')
axes[0].axhline(0.0, color='grey',  ls='--', lw=1.2, label='Uniform (0.0)')
axes[0].set_xlabel('Number of qubits (n)', fontsize=12)
axes[0].set_ylabel('Normalised XEB', fontsize=12)
axes[0].set_title('XEB vs n — one curve per N_train', fontsize=12)
axes[0].legend(fontsize=9, title='N_train')
axes[0].grid(True, alpha=0.3)
axes[0].set_xticks(n_values)

# Heatmap
heatmap = np.full((len(train_labels), len(n_values)), np.nan)
for j, n in enumerate(n_values):
    sub = df[df['n_bits'] == n]
    for i, label in enumerate(train_labels):
        for _, row in sub.iterrows():
            if poly_label(int(row['n_train']), n) == label:
                heatmap[i, j] = row['xeb_norm']; break

im = axes[1].imshow(heatmap, aspect='auto', cmap='RdYlGn', vmin=0, vmax=1, origin='upper')
axes[1].set_xticks(range(len(n_values)));     axes[1].set_xticklabels(n_values)
axes[1].set_yticks(range(len(train_labels))); axes[1].set_yticklabels(train_labels)
axes[1].set_xlabel('Number of qubits (n)', fontsize=12)
axes[1].set_ylabel('N_train', fontsize=12)
axes[1].set_title('Normalised XEB heatmap\n(green=ideal, red=uniform)', fontsize=12)
plt.colorbar(im, ax=axes[1], label='Normalised XEB')
for i in range(len(train_labels)):
    for j in range(len(n_values)):
        v = heatmap[i, j]
        if not np.isnan(v):
            axes[1].text(j, i, f'{v:.2f}', ha='center', va='center',
                         fontsize=7.5, color='black' if 0.2 < v < 0.8 else 'white')

plt.tight_layout()
plt.savefig('n_qubit_sweep.pdf')
plt.show()